In [10]:
import torch
from torchvision import datasets, transforms

# 定义数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),#将PIL.Image变换成torch.Tensor，并且将像素值归一化到[0,1]
])

# 下载并加载训练数据集
full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=False, transform=transform)

# 从训练集中分出5000样本作为验证集
train_size = len(full_train_dataset) - 5000
val_size = 5000
#random_split打乱样本
train_dataset, val_dataset = torch.utils.data.random_split(full_train_dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)  # batch_size是一个批次样本的数量
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)

# 下载并加载测试数据集
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=False, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
import torch.nn as nn

# 用nn.Sequential定义三层全连接网络
model = nn.Sequential(  
    nn.Flatten(),                 # 展平成一维向量 (batch_size, 28*28)
    nn.Linear(28*28, 512),
    nn.ReLU(),
    nn.Linear(512, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

In [12]:
import torch
import torch.optim as optim

# 判断设备：如果有GPU则用GPU，否则用CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. 设置交叉熵损失函数和SGD优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. 编写评估函数 evaluating
def evaluating(model, dataloader, device=device):
    model.eval()  # 将模型设置为评估模式（不启用dropout/BN等）
    correct = 0   # 记录预测正确的样本数量
    total = 0     # 记录总的样本数量
    with torch.no_grad():  # 在评估时不计算梯度，节省显存，提高速度
        for images, labels in dataloader:  # 遍历dataloader中的所有batch
            images = images.to(device)     # 将输入数据移动到指定设备（CPU或GPU）
            labels = labels.to(device)     # 将标签移动到指定设备
            outputs = model(images)        # 前向传播，获得模型输出
            predicted = torch.argmax(outputs, dim=1)  # 获得最大logits的类别索引
            total += labels.size(0)        # 累加本batch中的样本数量
            correct += (predicted == labels).sum().item()  # 累加预测正确的数量
    acc = 100 * correct / total           # 计算准确率
    return acc                           # 返回准确率

# 5. 编写训练函数 train
def train(model, trainloader, valloader, criterion, optimizer, epochs=10, device=device):
    model.to(device)  # 将模型移动到指定设备
    for epoch in range(epochs):  # 循环训练若干个epoch
        model.train()  # 设置为训练模式
        running_loss = 0.0  # 记录累计损失
        for batch_idx, (images, labels) in enumerate(trainloader):  # 遍历训练集的每个batch
            images = images.to(device)  # 将图片数据移动到设备
            labels = labels.to(device)  # 将标签数据移动到设备
            
            optimizer.zero_grad()       # 梯度清零
            outputs = model(images)     # 前向传播
            loss = criterion(outputs, labels)  # 计算损失
            loss.backward()             # 反向传播
            optimizer.step()            # 更新参数,w=w-lr*grad
            
            running_loss += loss.item() # 累加本batch的损失
            if (batch_idx + 1) % 100 == 0:  # 每隔100个batch打印一次损失
                print(f'Epoch [{epoch + 1}/{epochs}], Step [{batch_idx + 1}/{len(trainloader)}], Loss: {loss.item():.4f}')

        avg_loss = running_loss / len(trainloader)  # 计算平均损失
        train_acc = evaluating(model, trainloader, device)  # 计算训练集准确率
        val_acc = evaluating(model, valloader, device)    # 计算测试集准确率
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {avg_loss:.4f}, Train Acc: {train_acc:.2f}%, val Acc: {val_acc:.2f}%')

In [13]:
# 训练模型
num_epochs = 10  # 可以根据需要进行调节
train(model, train_loader, val_loader, criterion, optimizer, epochs=num_epochs, device=device)

Epoch [1/10], Step [100/860], Loss: 0.6084
Epoch [1/10], Step [200/860], Loss: 0.6120
Epoch [1/10], Step [300/860], Loss: 0.3720
Epoch [1/10], Step [400/860], Loss: 0.6767
Epoch [1/10], Step [500/860], Loss: 0.6408
Epoch [1/10], Step [600/860], Loss: 0.4615
Epoch [1/10], Step [700/860], Loss: 0.6655
Epoch [1/10], Step [800/860], Loss: 0.3801
Epoch [1/10], Loss: 0.5399, Train Acc: 84.21%, val Acc: 84.32%
Epoch [2/10], Step [100/860], Loss: 0.3640
Epoch [2/10], Step [200/860], Loss: 0.5833
Epoch [2/10], Step [300/860], Loss: 0.5142
Epoch [2/10], Step [400/860], Loss: 0.6038
Epoch [2/10], Step [500/860], Loss: 0.2599
Epoch [2/10], Step [600/860], Loss: 0.4024
Epoch [2/10], Step [700/860], Loss: 0.3828
Epoch [2/10], Step [800/860], Loss: 0.2547
Epoch [2/10], Loss: 0.4111, Train Acc: 84.91%, val Acc: 84.24%
Epoch [3/10], Step [100/860], Loss: 0.3996
Epoch [3/10], Step [200/860], Loss: 0.4792
Epoch [3/10], Step [300/860], Loss: 0.2537
Epoch [3/10], Step [400/860], Loss: 0.4067
Epoch [3/10], 